# 02 Classification: High Risk Flag


## Objective

在防止目标泄漏的前提下，补充分层交叉验证调参、PR-AUC 指标和训练集内验证阈值调优。最终测试集只用于一次评估。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import (
    CLASSIFICATION_TARGET,
    FIGURES_DIR,
    HIGH_RISK_LEAKAGE_COLUMNS,
    ID_COLUMNS,
    RANDOM_STATE,
    RESULTS_DIR,
)
from data_utils import ensure_project_dirs, load_processed_dataset
from feature_engineering import get_excluded_columns, make_feature_target
from model_utils import run_classification_experiment
from visualization import (
    plot_classification_metrics_comparison,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_precision_recall_curve,
    plot_roc_curve,
)

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析


## Leakage-Aware Feature Check

分类任务继续排除身心结果变量、效率变量、数字依赖分数、目标列和 ID。


In [2]:
df = load_processed_dataset(fallback_to_raw=True)
excluded = get_excluded_columns(task="classification")
reasons = []
for column in excluded:
    if column == CLASSIFICATION_TARGET:
        reasons.append("classification target")
    elif column in HIGH_RISK_LEAKAGE_COLUMNS:
        reasons.append("specified leakage/outcome column")
    elif column in ID_COLUMNS:
        reasons.append("identifier")
    else:
        reasons.append("excluded by configuration")

exclusion_table = pd.DataFrame({"excluded_column": excluded, "reason": reasons})
exclusion_table.to_csv(RESULTS_DIR / "classification_feature_exclusion.csv", index=False)

X, y = make_feature_target(df, task="classification")
print(f"Feature matrix shape: {X.shape}")
print(f"Target positive rate: {y.mean():.4f}")
display(exclusion_table)


Feature matrix shape: (3500, 22)
Target positive rate: 0.2014


,excluded_column,reason
0,anxiety_score,specified leakage/outcome column
1,depression_score,specified leakage/outcome column
2,digital_dependence_score,specified leakage/outcome column
3,focus_score,specified leakage/outcome column
4,happiness_score,specified leakage/outcome column
5,high_risk_flag,classification target
6,id,identifier
7,productivity_score,specified leakage/outcome column
8,stress_level,specified leakage/outcome column


## Tuned Classification Models and Threshold Selection

调参使用训练集内部的 StratifiedKFold；阈值使用训练集再划分出的 validation 集选择，避免在测试集上选择阈值。


In [3]:
classification_result = run_classification_experiment(df)
print(f"Selected probability model: {classification_result['best_model_name']}")
print(f"Final policy: {classification_result['final_policy']}, threshold={classification_result['final_threshold']:.2f}")
display(classification_result["tuned_metrics"])
display(classification_result["threshold_tuning"])


Selected probability model: gradient_boosting
Final policy: recall_at_least_60_best_precision, threshold=0.14


,dataset,model,threshold_policy,threshold,best_params,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc,tn,fp,fn,tp
0,validation,gradient_boosting,default_0_50,0.50,"{'model__n_estimators': 100, 'model__max_depth...",0.817352,0.642857,0.204545,0.310345,0.587987,0.723283,0.480507,510,15,105,27
1,validation,random_forest,default_0_50,0.50,"{'model__n_estimators': 400, 'model__min_sampl...",0.808219,0.535714,0.340909,0.416667,0.633312,0.709798,0.475859,486,39,87,45
2,validation,logistic_regression,default_0_50,0.50,"{'model__C': 0.01, 'model__class_weight': None}",0.811263,0.642857,0.136364,0.225000,0.558658,0.720087,0.454345,515,10,114,18
3,test,gradient_boosting,default_0_50,0.50,"{'model__n_estimators': 100, 'model__max_depth...",0.827429,0.681159,0.267045,0.383673,0.617786,0.753081,0.508394,677,22,129,47
4,test,gradient_boosting,max_f1,0.35,"{'model__n_estimators': 100, 'model__max_depth...",0.829714,0.595745,0.477273,0.529968,0.697864,0.753081,0.508394,642,57,92,84
5,test,gradient_boosting,recall_at_least_60_best_precision,0.14,"{'model__n_estimators': 100, 'model__max_depth...",0.776000,0.459350,0.642045,0.535545,0.725887,0.753081,0.508394,566,133,63,113
6,test,gradient_boosting,recall_at_least_70_best_precision,0.12,"{'model__n_estimators': 100, 'model__max_depth...",0.488000,0.253623,0.795455,0.384615,0.603021,0.753081,0.508394,287,412,36,140


,dataset,model,threshold,policy,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc,tn,fp,fn,tp,best_params
0,validation,gradient_boosting,0.50,default_0_50,0.817352,0.642857,0.204545,0.310345,0.587987,0.723283,0.480507,510,15,105,27,"{'model__n_estimators': 100, 'model__max_depth..."
1,validation,gradient_boosting,0.35,max_f1,0.824962,0.575221,0.492424,0.530612,0.700498,0.723283,0.480507,477,48,67,65,"{'model__n_estimators': 100, 'model__max_depth..."
2,validation,gradient_boosting,0.14,recall_at_least_60_best_precision,0.750381,0.417526,0.613636,0.496933,0.699199,0.723283,0.480507,412,113,51,81,"{'model__n_estimators': 100, 'model__max_depth..."
3,validation,gradient_boosting,0.12,recall_at_least_70_best_precision,0.520548,0.263566,0.772727,0.393064,0.614935,0.723283,0.480507,240,285,30,102,"{'model__n_estimators': 100, 'model__max_depth..."


## Classification Figures

这些图用于支持模型选择和阈值解释：指标对比、混淆矩阵、ROC 曲线、PR 曲线和置换重要性。


In [4]:
plot_classification_metrics_comparison(
    classification_result["tuned_metrics"],
    FIGURES_DIR / "classification_tuned_metrics_comparison.png",
)
plot_confusion_matrix(
    classification_result["confusion_matrix"],
    FIGURES_DIR / "classification_final_confusion_matrix.png",
)
plot_roc_curve(
    classification_result["y_test"],
    classification_result["y_score"],
    FIGURES_DIR / "classification_roc_curve.png",
)
plot_precision_recall_curve(
    classification_result["y_test"],
    classification_result["y_score"],
    FIGURES_DIR / "classification_precision_recall_curve.png",
)
plot_feature_importance(
    classification_result["feature_importance"],
    FIGURES_DIR / "classification_permutation_importance.png",
    title="Classification Permutation Importance",
)
classification_result["feature_importance"].head(15)


,feature,importance_mean,importance_std
0,device_hours_per_day,0.102618,0.024889
1,sleep_hours,0.032097,0.008048
2,device_to_sleep_ratio,0.005526,0.012394
3,social_to_study_ratio,0.004037,0.003577
4,social_media_hours,0.003434,0.002208
5,region,0.002727,0.004675
6,notifications_per_device_hour,0.002258,0.002328
7,study_hours,0.000589,0.001461
8,study_mins,0.000087,0.002272
9,physical_activity_days,0.000000,0.000000


<Figure size 800x640 with 0 Axes>

<Figure size 800x640 with 0 Axes>